# Failure-Aware RAG — Interactive Exploration

This notebook walks through every stage of the pipeline:
1. **Retrieval** — Encode query, search FAISS index
2. **Generation** — Build prompt, run LLM (or mock)
3. **Claim Extraction** — Split answer into atomic facts
4. **NLI Verification** — Label each claim against evidence
5. **Metric Computation** — SCR, CR, TVE, CDEE, CHS
6. **Training Signals** — DPO pairs, rejection filtering

> **No GPU required** — cells marked `[CPU-safe]` run on any machine.


In [ ]:
import sys
from pathlib import Path

# Make src/ importable from notebooks/
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

## 1. Claim Extraction [CPU-safe]

The `SpacyClaimExtractor` splits an answer into atomic verifiable claims.  
If spaCy is unavailable, it falls back to regex sentence splitting.

In [ ]:
from src.diagnosis.claim_extractor import SpacyClaimExtractor

answer = (
    "The Eiffel Tower is located in Paris, France. "
    "It was built by Gustave Eiffel between 1887 and 1889. "
    "Currently it is the most visited paid monument in the world."
)

extractor = SpacyClaimExtractor()
claims = extractor.extract(answer)
print(f"Extracted {len(claims)} claims:")
for i, c in enumerate(claims, 1):
    print(f"  {i}. {c}")

## 2. Temporal Claim Detection [CPU-safe]

Claims containing temporal keywords automatically get `is_temporal=True`,
making them candidates for TVE (Temporal Validity Error).

In [ ]:
from src.diagnosis.verification_engine import _is_temporal_claim

test_claims = [
    "The current president of France is Emmanuel Macron.",
    "As of 2024, the population of Earth is 8 billion.",
    "The Eiffel Tower was built in 1889.",
    "Water boils at 100 degrees Celsius.",
    "Recently, scientists discovered a new exoplanet.",
]

for c in test_claims:
    print(f"  {'TEMPORAL' if _is_temporal_claim(c) else 'static  '}: {c}")

## 3. Hallucination Metrics [CPU-safe]

The `MetricEngine` takes a list of `ClaimVerificationResult` objects and computes:
- **SCR** — Support Coverage Ratio (↑ better)
- **CR** — Conflict Rate (↓ better)
- **TVE** — Temporal Validity Error (↓ better)
- **CDEE** — Cross-Document Entailment Error (↓ better)
- **CHS** — Composite Hallucination Score (↓ better)

In [ ]:
from src.diagnosis.metric_engine import MetricEngine
from src.diagnosis.verification_engine import ClaimVerificationResult, NLILabel

def make_result(label, is_temporal=False, is_outdated=False):
    return ClaimVerificationResult(
        claim="example claim",
        label=label,
        entailment_score=1.0 if label == NLILabel.ENTAILMENT else 0.0,
        contradiction_score=1.0 if label == NLILabel.CONTRADICTION else 0.0,
        neutral_score=1.0 if label == NLILabel.NEUTRAL else 0.0,
        is_temporal=is_temporal,
        is_outdated=is_outdated,
    )

engine = MetricEngine()

# --- Perfect factual answer ---
good_results = [make_result(NLILabel.ENTAILMENT) for _ in range(5)]
m_good = engine.compute(good_results)
print("Perfect answer:", m_good.summary())

# --- Hallucinated answer ---
bad_results = [
    make_result(NLILabel.ENTAILMENT),
    make_result(NLILabel.ENTAILMENT),
    make_result(NLILabel.CONTRADICTION),
    make_result(NLILabel.NEUTRAL, is_temporal=True, is_outdated=True),
]
m_bad = engine.compute(bad_results)
print("Hallucinated:  ", m_bad.summary())

## 4. QA Metrics [CPU-safe]

Standard token-level F1 and exact match (EM) used for QA evaluation.

In [ ]:
from src.evaluation.evaluator import f1_score, exact_match

examples = [
    ("Paris", "Paris"),
    ("the Eiffel Tower", "Eiffel Tower"),
    ("Alexander Graham Bell", "Bell"),
    ("The crisis caused the banks to fail", "subprime mortgages"),
]

print(f"{'Prediction':<35} {'GT':<25} {'F1':>6} {'EM':>5}")
print("-" * 75)
for pred, gt in examples:
    f1 = f1_score(pred, gt)
    em = exact_match(pred, gt)
    print(f"{pred:<35} {gt:<25} {f1:>6.3f} {em:>5.1f}")

## 5. Full Mock Demo [CPU-safe]

Run the complete end-to-end pipeline using mock components.  
The Eiffel Tower query deliberately includes a Napoleon hallucination to demonstrate detection.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, str(repo_root / "scripts" / "demo.py"), "--verbose"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

## 6. Training Signal Generation [CPU-safe]

Shows how `TrainingSignalGenerator` converts `DiagnosisOutput` objects  
into `TrainingSample` objects and how DPO pairs are formed.

In [ ]:
from src.training.qlora_trainer import TrainingSample, prepare_dpo_dataset

# Simulated samples — same query answered twice with different quality
samples = [
    TrainingSample(query="Who built the Eiffel Tower?",
                   answer="Gustave Eiffel built the tower in 1889.",
                   chs=0.05, prompt="[P]"),
    TrainingSample(query="Who built the Eiffel Tower?",
                   answer="Napoleon commissioned the tower in 1800.",
                   chs=0.72, prompt="[P]"),
    TrainingSample(query="What is DNA?",
                   answer="DNA carries genetic information.",
                   chs=0.08, prompt="[P2]"),
]

pairs = prepare_dpo_dataset(samples, min_gap=0.2)
print(f"DPO pairs created: {len(pairs)}")
for p in pairs:
    print(f"  gap={p.chs_gap:.2f}")
    print(f"    chosen  : {p.chosen}")
    print(f"    rejected: {p.rejected}")

## 7. Building the Index (requires sentence-transformers)

Run this cell only after `pip install sentence-transformers faiss-cpu`.

```bash
make index N_DOCS=1000
```

Or inline:

In [ ]:
# from src.retrieval.encoder import DenseEncoder, build_index_from_corpus
# encoder = DenseEncoder("sentence-transformers/all-MiniLM-L6-v2")
# index = build_index_from_corpus("data/corpus.jsonl", encoder, "data/faiss.index")
# print(f"Index built with {index.index.ntotal:,} vectors")
print("Uncomment the lines above after installing sentence-transformers + faiss-cpu.")

## 8. Results Visualisation (requires matplotlib)

After running an evaluation (`make evaluate`), generate charts:

```bash
make visualise
```

Or inline after `pip install matplotlib`:

In [ ]:
# from scripts.visualise_results import plot_chs_distribution
# import matplotlib
# matplotlib.use('inline')
# plot_chs_distribution([0.1, 0.2, 0.05, 0.7, 0.3, 0.55, 0.8, 0.12], model_tag="demo")
print("Uncomment the lines above after installing matplotlib.")